In [ ]:
import json
import numpy as np
import os

# CONFIGURE THESE PATHS TO MATCH YOUR FOLDERS
EMBEDDINGS_DIR = "../../target/embeddings/tct_colbert-v2-hnp-msmarco"
# Note: If your actual embeddings binary file is named "index" instead of "embedding", change the line below:
EMBEDDING_BIN = os.path.join(EMBEDDINGS_DIR, "index") 
DOCID_FILE = os.path.join(EMBEDDINGS_DIR, "docid")
OUTPUT_FILE = os.path.join(EMBEDDINGS_DIR, "embeddings.jsonl")

# 1. Read the docids
with open(DOCID_FILE, 'r') as f:
    docids = [line.strip() for line in f if line.strip()]

print(f"Found {len(docids)} document IDs.")

# 2. Load the binary embeddings (TCT-ColBERT output is usually float32 vectors)
try:
    embeddings = np.fromfile(EMBEDDING_BIN, dtype=np.float32)
except FileNotFoundError:
    print(f"Error: Could not find the binary file at {EMBEDDING_BIN}")
    print("Note: Your binary file might be named 'index' instead of 'embedding'. Rename it or update the script.")
    exit(1)

# 3. Reshape the data (768 is the standard dimension for this TCT-ColBERT model)
embedding_dim = 768
expected_size = len(docids) * embedding_dim

if len(embeddings) != expected_size:
    print(f"Warning: Binary file size mismatch. Expected {expected_size} floats, got {len(embeddings)}.")
    print("If you have fewer embeddings, please check your data.")
    # Attempt to reshape anyway, but it may fail if dims don't match.
    try:
        embeddings = embeddings.reshape(-1, embedding_dim)
    except ValueError:
        print("Reshape failed. Please check your embedding file and docid file match exactly.")
        exit(1)
else:
    embeddings = embeddings.reshape(len(docids), embedding_dim)

# 4. Write to JSONL
print(f"Writing JSONL file to {OUTPUT_FILE}...")
with open(OUTPUT_FILE, 'w') as f:
    for i, docid in enumerate(docids):
        # Convert numpy array to standard python list for JSON
        json.dump({"id": docid, "vector": embeddings[i].tolist()}, f)
        f.write('\n')

print("Done! You can now use the new pyserini module.")

Found 244 document IDs.
Error: Could not find the binary file at ../../target/embeddings/tct_colbert-v2-hnp-msmarco\embedding
Note: Your binary file might be named 'index' instead of 'embedding'. Rename it or update the script.


NameError: name 'embeddings' is not defined